Step 1 — Inspect the Schema of Bronze People dimension

In [0]:
bronze_people = spark.table(
    "workforce_analytics.bronze.dim_people_raw"
)

bronze_people.printSchema()

display(bronze_people.limit(10))

Step 1: Create the cleaned DimPeople table
We'll:
- Trim whitespace from text columns.
- Convert hire_date from a string into a proper date.
- Keep _source_file and _ingested_at for data lineage.
- Keep people_id as a string, so we don't accidentally change its format.

In [0]:
from pyspark.sql import functions as F

# Read Bronze table
bronze_people = spark.table(
    "workforce_analytics.bronze.dim_people_raw"
)

# Clean the data
silver_people = bronze_people.select(
    F.trim(F.col("people_id")).alias("people_id"),
    F.trim(F.col("first_name")).alias("first_name"),
    F.trim(F.col("surname")).alias("surname"),
    F.trim(F.col("employee_name")).alias("employee_name"),
    F.trim(F.col("job_role")).alias("job_role"),
    F.trim(F.col("employment_type")).alias("employment_type"),
    F.trim(F.col("team")).alias("team"),
    F.to_date(F.trim(F.col("hire_date")), "yyyy-MM-dd").alias("hire_date"),
    F.col("_source_file"),
    F.col("_ingested_at")
)

In [0]:
# Check the cleaned schema
silver_people.printSchema()

# View sample records
display(silver_people.limit(10))

# Check total rows
print("Total rows:", silver_people.count())

# Check for missing people IDs
print(
    "Rows with missing people_id:",
    silver_people.filter(
        F.col("people_id").isNull() | (F.col("people_id") == "")
    ).count()
)

# Check for duplicate people IDs
display(
    silver_people
    .groupBy("people_id")
    .count()
    .filter(F.col("count") > 1)
)

In [0]:
display(
    bronze_people.select("hire_date")
    .distinct()
    .limit(20)
)

In [0]:
# Check for duplicate people IDs
duplicate_people = (
    silver_people
    .groupBy("people_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate people IDs:", duplicate_people.count())

display(duplicate_people)

In [0]:
silver_people.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workforce_analytics.silver.dim_people")

# Then validate the saved table:
saved_people = spark.table(
    "workforce_analytics.silver.dim_people"
)

print("Silver DimPeople rows:", saved_people.count())

display(saved_people.limit(10))